In [ ]:
%pip -q install datasets scikit-learn pandas matplotlib joblib transformers accelerate

In [ ]:
import os, re, time, json, math, random, copy, collections, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, classification_report,
    confusion_matrix, log_loss
)

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| PyTorch:', torch.__version__)

# Dataset / label settings (kept compatible with the supplied TTS notebook)
DATASET_ID = 'google-research-datasets/go_emotions'
DATASET_CONFIG = 'simplified'
TXT_CLASSES = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise']
CLASS_TO_ID = {name: i for i, name in enumerate(TXT_CLASSES)}
MAX_LEN = 48
MAX_VOCAB = 15000                # including 2 reserved IDs: PAD=0, UNK=1
BATCH_SIZE = 64
CUSTOM_EPOCHS = 8
CUSTOM_LR = 5e-4
CUSTOM_PATIENCE = 2

# The pretrained baseline downloads distilroberta-base (~hundreds of MB).
RUN_DISTILROBERTA = True
HF_MODEL_NAME = 'distilroberta-base'
HF_MAX_LEN = 64
HF_BATCH_SIZE = 16 if DEVICE.type == 'cuda' else 8
HF_EPOCHS = 3
HF_PATIENCE = 1

OUTPUT_DIR = Path('classifier_comparison_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR.resolve())

In [ ]:
EKMAN = {
    'anger': ['anger', 'annoyance', 'disapproval'],
    'disgust': ['disgust'],
    'fear': ['fear', 'nervousness'],
    'joy': ['joy', 'amusement', 'approval', 'excitement', 'gratitude', 'love', 'optimism', 'relief', 'pride', 'admiration', 'desire', 'caring'],
    'sadness': ['sadness', 'disappointment', 'embarrassment', 'grief', 'remorse'],
    'surprise': ['surprise', 'realization', 'confusion', 'curiosity'],
}
label_to_class = {label: cls for cls, labels in EKMAN.items() for label in labels}
label_to_class['neutral'] = 'neutral'

raw_ds = load_dataset(DATASET_ID, DATASET_CONFIG)
label_names = raw_ds['train'].features['labels'].feature.names
unknown_names = sorted(set(label_names) - set(label_to_class))
if unknown_names:
    raise ValueError(f'Labels are not mapped: {unknown_names}. Review label_to_class before training.')

filtered_data = {}
filter_report = []
for raw_split, split_name in [('train', 'train'), ('validation', 'validation'), ('test', 'test')]:
    rows, mixed_or_empty = [], 0
    for example in raw_ds[raw_split]:
        raw_labels = [label_names[i] for i in example['labels']]
        mapped = {label_to_class[x] for x in raw_labels}
        if len(mapped) == 1:
            label = next(iter(mapped))
            rows.append((str(example['text']), label))
        else:
            mixed_or_empty += 1
    filtered_data[split_name] = rows
    filter_report.append({'split': split_name, 'kept': len(rows), 'dropped_mixed': mixed_or_empty})

print(pd.DataFrame(filter_report).to_string(index=False))
train_texts = [x for x, _ in filtered_data['train']]
train_labels = np.array([CLASS_TO_ID[y] for _, y in filtered_data['train']], dtype=np.int64)
val_texts = [x for x, _ in filtered_data['validation']]
val_labels = np.array([CLASS_TO_ID[y] for _, y in filtered_data['validation']], dtype=np.int64)
test_texts = [x for x, _ in filtered_data['test']]
test_labels = np.array([CLASS_TO_ID[y] for _, y in filtered_data['test']], dtype=np.int64)

class_counts = pd.DataFrame({
    split: pd.Series(collections.Counter(label for _, label in rows)).reindex(TXT_CLASSES, fill_value=0)
    for split, rows in filtered_data.items()
})
display(class_counts)
ax = class_counts.plot(kind='bar', figsize=(11, 4))
ax.set_title('Class distribution after mapping/filtering')
ax.set_ylabel('Examples'); ax.set_xlabel('Emotion'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
def metrics_for(y_true, probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    y_pred = probabilities.argmax(axis=1)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=np.arange(len(TXT_CLASSES)), average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, labels=np.arange(len(TXT_CLASSES)), average='weighted', zero_division=0),
        'multiclass_brier': float(np.mean(np.sum((probabilities - np.eye(len(TXT_CLASSES))[y_true]) ** 2, axis=1))),
    }

def report_model(name, y_true, probabilities, split='test'):
    row = {'model': name, 'split': split, **metrics_for(y_true, probabilities)}
    print(f'\n{name} — {split}')
    print(classification_report(y_true, probabilities.argmax(axis=1), labels=np.arange(len(TXT_CLASSES)), target_names=TXT_CLASSES, zero_division=0, digits=3))
    return row

all_val_probs = {}
all_test_probs = {}
metric_rows = []

In [ ]:
tfidf_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True, strip_accents='unicode', ngram_range=(1, 2),
        min_df=2, max_features=200_000, sublinear_tf=True,
    )),
    ('clf', LogisticRegression(
        max_iter=1000, class_weight='balanced', solver='lbfgs', random_state=SEED,
    )),
])
start = time.perf_counter()
tfidf_model.fit(train_texts, train_labels)
print(f'Training time: {(time.perf_counter()-start)/60:.2f} minutes')
val_probs = tfidf_model.predict_proba(val_texts)
test_probs = tfidf_model.predict_proba(test_texts)
all_val_probs['TF-IDF + Logistic Regression'] = val_probs
all_test_probs['TF-IDF + Logistic Regression'] = test_probs
metric_rows.append(report_model('TF-IDF + Logistic Regression', val_labels, val_probs, 'validation'))
metric_rows.append(report_model('TF-IDF + Logistic Regression', test_labels, test_probs, 'test'))
joblib.dump(tfidf_model, OUTPUT_DIR / 'tfidf_logistic.joblib')

In [ ]:
WORD_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?.,]")
def tok(text):
    return WORD_RE.findall(str(text).lower())

word_counts = collections.Counter(w for text in train_texts for w in tok(text))
clf_words = [w for w, _ in word_counts.most_common(MAX_VOCAB - 2)]
CLF_W2I = {w: i + 2 for i, w in enumerate(clf_words)}  # PAD=0, UNK=1


def clf_encode(text, maxlen=MAX_LEN):
    ids = [CLF_W2I.get(w, 1) for w in tok(text)][:maxlen]
    return ids or [1]

def make_xy(texts, labels, maxlen=MAX_LEN):
    X = torch.zeros((len(texts), maxlen), dtype=torch.long)
    for i, text in enumerate(texts):
        ids = clf_encode(text, maxlen)
        X[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    return X, torch.tensor(labels, dtype=torch.long)

X_train, y_train = make_xy(train_texts, train_labels)
X_val, y_val = make_xy(val_texts, val_labels)
X_test, y_test = make_xy(test_texts, test_labels)
print('Vocabulary:', len(clf_words), '| tensor shapes:', X_train.shape, X_val.shape, X_test.shape)

# The following blocks mirror the class definitions in the uploaded TTS notebook.
def sinusoid(T, d, device):
    pos = torch.arange(T, device=device).float()[:, None]
    i = torch.arange(0, d, 2, device=device).float()[None]
    ang = pos / (10000 ** (i / d))
    pe = torch.zeros(T, d, device=device)
    pe[:, 0::2] = torch.sin(ang)
    pe[:, 1::2] = torch.cos(ang)
    return pe

class Block(nn.Module):
    """Pre-LN self-attention + convolutional feed-forward block; True in mask means valid token."""
    def __init__(self, d, h, ffn, sd=0, p=0.1):
        super().__init__()
        self.sd, self.h = sd, h
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)
        self.c1 = nn.Conv1d(d, ffn, 3, padding=1)
        self.c2 = nn.Conv1d(ffn, d, 1)
        self.dp = nn.Dropout(p)
    def forward(self, x, mask, sty=None):
        B, T, D = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask[:, None, None, :])
        x = x + self.dp(self.o(y.transpose(1, 2).reshape(B, T, D)))
        mm = mask.unsqueeze(-1).to(x.dtype)
        h = (self.n2(x) * mm).transpose(1, 2)
        x = x + self.dp(self.c2(F.relu(self.c1(h))).transpose(1, 2))
        return x * mm

class EmoText(nn.Module):
    """Same module names and dimensions as the custom classifier in the supplied TTS notebook."""
    def __init__(self, V, n, d=128, h=4, ffn=256, L=2, p=0.15):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(V, d, padding_idx=0)
        nn.init.normal_(self.emb.weight, 0, d ** -0.5)
        self.blocks = nn.ModuleList([Block(d, h, ffn, 0, p) for _ in range(L)])
        self.ln = nn.LayerNorm(d)
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(d, n)
    def forward(self, ids):
        mask = ids.ne(0)
        x = self.emb(ids) * math.sqrt(self.d) + sinusoid(ids.shape[1], self.d, ids.device)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln(x)
        mm = mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * mm).sum(1) / mm.sum(1).clamp(min=1)
        return self.fc(self.drop(pooled))

custom_model = EmoText(len(clf_words) + 2, len(TXT_CLASSES)).to(DEVICE)
print(f'Custom model parameters: {sum(p.numel() for p in custom_model.parameters()):,}')

Vocabulary: 14998 | tensor shapes: torch.Size([39555, 48]) torch.Size([4946, 48]) torch.Size([4968, 48])
Custom model parameters: 2,317,191


In [ ]:
def predict_custom_probabilities(model, X, batch_size=512):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            logits = model(X[start:start + batch_size].to(DEVICE))
            outputs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(outputs, axis=0)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
class_counts_train = torch.bincount(y_train, minlength=len(TXT_CLASSES)).float()
class_weights = (class_counts_train.sum() / class_counts_train.clamp(min=1)).sqrt()
class_weights = (class_weights / class_weights.mean()).to(DEVICE)
optimizer = torch.optim.AdamW(custom_model.parameters(), lr=CUSTOM_LR, weight_decay=1e-2)

best_val_f1 = -1.0
best_epoch = 0
best_step = 0
best_state = None
best_optimizer_state = None
bad_epochs = 0
history = []
global_step = 0

for epoch in range(1, CUSTOM_EPOCHS + 1):
    custom_model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        # Word dropout on non-padding tokens; used only for training.
        drop_mask = (torch.rand(xb.shape, device=DEVICE) < 0.10) & (xb != 0)
        xb_aug = xb.masked_fill(drop_mask, 1)
        logits = custom_model(xb_aug)
        loss = F.cross_entropy(logits, yb, weight=class_weights, label_smoothing=0.05)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(custom_model.parameters(), 1.0)
        optimizer.step()
        running_loss += float(loss.item())
        global_step += 1

    val_probs_epoch = predict_custom_probabilities(custom_model, X_val)
    val_scores = metrics_for(val_labels, val_probs_epoch)
    epoch_row = {'epoch': epoch, 'train_loss': running_loss / max(1, len(train_loader)), **val_scores}
    history.append(epoch_row)
    print(f"Epoch {epoch:02d} | loss={epoch_row['train_loss']:.4f} | val macro-F1={val_scores['macro_f1']:.4f} | val acc={val_scores['accuracy']:.4f}")

    if val_scores['macro_f1'] > best_val_f1:
        best_val_f1 = val_scores['macro_f1']
        best_epoch = epoch
        best_step = global_step
        best_state = {k: v.detach().cpu().clone() for k, v in custom_model.state_dict().items()}
        best_optimizer_state = copy.deepcopy(optimizer.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= CUSTOM_PATIENCE:
            print('Early stopping on validation macro-F1.')
            break

custom_model.load_state_dict(best_state)
optimizer.load_state_dict(best_optimizer_state)
custom_model.eval()
custom_history = pd.DataFrame(history)
display(custom_history)
custom_history.to_csv(OUTPUT_DIR / 'custom_transformer_training_history.csv', index=False)

val_probs = predict_custom_probabilities(custom_model, X_val)
test_probs = predict_custom_probabilities(custom_model, X_test)
all_val_probs['Tiny Transformer (from scratch)'] = val_probs
all_test_probs['Tiny Transformer (from scratch)'] = test_probs
metric_rows.append(report_model('Tiny Transformer (from scratch)', val_labels, val_probs, 'validation'))
metric_rows.append(report_model('Tiny Transformer (from scratch)', test_labels, test_probs, 'test'))

Epoch 01 | loss=1.5784 | val macro-F1=0.5733 | val acc=0.6506
Epoch 02 | loss=1.3174 | val macro-F1=0.5604 | val acc=0.6359
Epoch 03 | loss=1.1781 | val macro-F1=0.5597 | val acc=0.6401
Early stopping on validation macro-F1.


,epoch,train_loss,accuracy,balanced_accuracy,macro_f1,weighted_f1,multiclass_brier
0,1,1.578439,0.650627,0.582609,0.573278,0.653925,0.491723
1,2,1.317435,0.635867,0.606932,0.560359,0.644550,0.502823
2,3,1.178133,0.640113,0.583792,0.559653,0.640403,0.488284



Tiny Transformer (from scratch) — validation
              precision    recall  f1-score   support

     neutral      0.596     0.637     0.616      1592
         joy      0.827     0.754     0.789      1941
     sadness      0.621     0.549     0.583       266
       anger      0.468     0.431     0.448       555
        fear      0.583     0.583     0.583        72
     disgust      0.403     0.508     0.449        61
    surprise      0.488     0.617     0.545       459

    accuracy                          0.651      4946
   macro avg      0.569     0.583     0.573      4946
weighted avg      0.661     0.651     0.654      4946


Tiny Transformer (from scratch) — test
              precision    recall  f1-score   support

     neutral      0.586     0.632     0.608      1606
         joy      0.804     0.748     0.775      1863
     sadness      0.586     0.516     0.549       283
       anger      0.472     0.420     0.444       572
        fear      0.625     0.625     0.625   

In [ ]:
hf_model = None
hf_tokenizer = None

if RUN_DISTILROBERTA:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from torch.optim import AdamW

    hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
    hf_model = AutoModelForSequenceClassification.from_pretrained(
        HF_MODEL_NAME,
        num_labels=len(TXT_CLASSES),
        id2label={i: label for i, label in enumerate(TXT_CLASSES)},
        label2id={label: i for i, label in enumerate(TXT_CLASSES)},
    ).to(DEVICE)

    def hf_batches(texts, labels=None, batch_size=HF_BATCH_SIZE, shuffle=False):
        # Tokenize in the collator so the full tokenized corpus is not held in memory.
        class TextDataset(torch.utils.data.Dataset):
            def __len__(self): return len(texts)
            def __getitem__(self, idx):
                return texts[idx], (int(labels[idx]) if labels is not None else -1)
        def collate(batch):
            texts_b, labels_b = zip(*batch)
            toks = hf_tokenizer(list(texts_b), truncation=True, padding=True, max_length=HF_MAX_LEN, return_tensors='pt')
            toks['labels'] = torch.tensor(labels_b, dtype=torch.long)
            return toks
        return DataLoader(TextDataset(), batch_size=batch_size, shuffle=shuffle, collate_fn=collate, num_workers=0)

    hf_train_loader = hf_batches(train_texts, train_labels, shuffle=True)
    hf_val_loader = hf_batches(val_texts, val_labels, shuffle=False)
    hf_test_loader = hf_batches(test_texts, test_labels, shuffle=False)
    hf_optimizer = AdamW(hf_model.parameters(), lr=2e-5, weight_decay=0.01)

    @torch.no_grad()
    def predict_hf_loader(model, loader):
        model.eval(); out = []
        for batch in loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
            logits = model(**inputs).logits
            out.append(torch.softmax(logits.float(), -1).cpu().numpy())
        return np.concatenate(out, axis=0)

    best_hf_f1, best_hf_epoch, bad_hf_epochs = -1.0, 0, 0
    best_hf_state = None
    hf_history = []
    for epoch in range(1, HF_EPOCHS + 1):
        hf_model.train(); losses = []
        for batch in hf_train_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = hf_model(**inputs).loss
            hf_optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(hf_model.parameters(), 1.0)
            hf_optimizer.step()
            losses.append(float(loss.item()))
        val_probs_epoch = predict_hf_loader(hf_model, hf_val_loader)
        val_scores = metrics_for(val_labels, val_probs_epoch)
        hf_history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), **val_scores})
        print(f"DistilRoBERTa epoch {epoch}/{HF_EPOCHS} | loss={np.mean(losses):.4f} | val macro-F1={val_scores['macro_f1']:.4f} | val acc={val_scores['accuracy']:.4f}")
        if val_scores['macro_f1'] > best_hf_f1:
            best_hf_f1 = val_scores['macro_f1']; best_hf_epoch = epoch; bad_hf_epochs = 0
            best_hf_state = {k: v.detach().cpu().clone() for k, v in hf_model.state_dict().items()}
        else:
            bad_hf_epochs += 1
            if bad_hf_epochs >= HF_PATIENCE:
                print('Early stopping DistilRoBERTa on validation macro-F1.')
                break

    hf_model.load_state_dict(best_hf_state)
    pd.DataFrame(hf_history).to_csv(OUTPUT_DIR / 'distilroberta_training_history.csv', index=False)
    val_probs = predict_hf_loader(hf_model, hf_val_loader)
    test_probs = predict_hf_loader(hf_model, hf_test_loader)
    all_val_probs['DistilRoBERTa (fine-tuned)'] = val_probs
    all_test_probs['DistilRoBERTa (fine-tuned)'] = test_probs
    metric_rows.append(report_model('DistilRoBERTa (fine-tuned)', val_labels, val_probs, 'validation'))
    metric_rows.append(report_model('DistilRoBERTa (fine-tuned)', test_labels, test_probs, 'test'))
else:
    print('Skipped DistilRoBERTa because RUN_DISTILROBERTA=False.')

In [ ]:
metrics_df = pd.DataFrame(metric_rows)
comparison_test = (metrics_df[metrics_df['split'] == 'test']
                   .sort_values('macro_f1', ascending=False)
                   .reset_index(drop=True))
display(comparison_test.round(4))
metrics_df.to_csv(OUTPUT_DIR / 'classifier_metrics_all_splits.csv', index=False)
comparison_test.to_csv(OUTPUT_DIR / 'classifier_comparison_test.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(comparison_test['model'], comparison_test['macro_f1'])
ax.set_ylim(0, 1); ax.set_ylabel('Test macro-F1'); ax.set_title('Text emotion classifier comparison')
ax.tick_params(axis='x', rotation=20)
for tick in ax.get_xticklabels(): tick.set_ha('right')
plt.tight_layout(); plt.show()

In [ ]:
for name, probs in all_test_probs.items():
    pred = probs.argmax(axis=1)
    cm = confusion_matrix(test_labels, pred, labels=np.arange(len(TXT_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(f'Confusion matrix — {name}')
    ax.set_xlabel('Predicted emotion'); ax.set_ylabel('Gold emotion')
    ax.set_xticks(range(len(TXT_CLASSES)), TXT_CLASSES, rotation=45, ha='right')
    ax.set_yticks(range(len(TXT_CLASSES)), TXT_CLASSES)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

# Inspect misclassifications from the custom model.
custom_name = 'Tiny Transformer (from scratch)'
if custom_name in all_test_probs:
    p = all_test_probs[custom_name]
    pred = p.argmax(axis=1)
    wrong_idx = np.where(pred != test_labels)[0]
    ranked = sorted(wrong_idx, key=lambda i: float(p[i].max()), reverse=True)[:30]
    errors = pd.DataFrame([{
        'text': test_texts[i], 'gold': TXT_CLASSES[test_labels[i]],
        'predicted': TXT_CLASSES[pred[i]], 'confidence': float(p[i].max())
    } for i in ranked])
    display(errors)
    errors.to_csv(OUTPUT_DIR / 'custom_transformer_high_confidence_errors.csv', index=False)

In [ ]:
BENCH_N = min(512, len(test_texts))
bench_texts = test_texts[:BENCH_N]
latency_rows = []

# TF-IDF: prediction includes vectorization.
t0 = time.perf_counter(); _ = tfidf_model.predict_proba(bench_texts); elapsed = time.perf_counter() - t0
latency_rows.append({'model': 'TF-IDF + Logistic Regression', 'ms_per_text': elapsed * 1000 / len(bench_texts)})

# Tiny Transformer: includes our regex tokenization/encoding and batched model forward.
def custom_predict_raw(texts, batch_size=256):
    X, _ = make_xy(texts, np.zeros(len(texts), dtype=np.int64))
    return predict_custom_probabilities(custom_model, X, batch_size=batch_size)
t0 = time.perf_counter(); _ = custom_predict_raw(bench_texts); elapsed = time.perf_counter() - t0
latency_rows.append({'model': 'Tiny Transformer (from scratch)', 'ms_per_text': elapsed * 1000 / len(bench_texts)})

# DistilRoBERTa: tokenizer + batched model inference.
if RUN_DISTILROBERTA and hf_model is not None:
    @torch.no_grad()
    def hf_predict_raw(texts, batch_size=HF_BATCH_SIZE):
        hf_model.eval(); outs = []
        for start in range(0, len(texts), batch_size):
            toks = hf_tokenizer(texts[start:start+batch_size], truncation=True, padding=True, max_length=HF_MAX_LEN, return_tensors='pt')
            toks = {k: v.to(DEVICE) for k, v in toks.items()}
            outs.append(torch.softmax(hf_model(**toks).logits.float(), -1).cpu().numpy())
        return np.concatenate(outs)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter(); _ = hf_predict_raw(bench_texts)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    latency_rows.append({'model': 'DistilRoBERTa (fine-tuned)', 'ms_per_text': elapsed * 1000 / len(bench_texts)})

latency_df = pd.DataFrame(latency_rows)
display(latency_df.round(3))
latency_df.to_csv(OUTPUT_DIR / 'classifier_latency.csv', index=False)

# Approximate model artifact size in MB (serializing model state is measured below).
torch.save(custom_model.state_dict(), OUTPUT_DIR / 'tiny_transformer_state_dict.pt')
size_rows = [
    {'model': 'TF-IDF + Logistic Regression', 'artifact_MB': (OUTPUT_DIR / 'tfidf_logistic.joblib').stat().st_size / (1024**2)},
    {'model': 'Tiny Transformer (from scratch)', 'artifact_MB': (OUTPUT_DIR / 'tiny_transformer_state_dict.pt').stat().st_size / (1024**2)},
]
if RUN_DISTILROBERTA and hf_model is not None:
    hf_dir = OUTPUT_DIR / 'distilroberta_finetuned'
    hf_dir.mkdir(exist_ok=True)
    hf_model.save_pretrained(hf_dir)
    hf_tokenizer.save_pretrained(hf_dir)
    folder_bytes = sum(p.stat().st_size for p in hf_dir.rglob('*') if p.is_file())
    size_rows.append({'model': 'DistilRoBERTa (fine-tuned)', 'artifact_MB': folder_bytes / (1024**2)})
size_df = pd.DataFrame(size_rows)
display(size_df.round(3))
size_df.to_csv(OUTPUT_DIR / 'classifier_artifact_sizes.csv', index=False)

In [ ]:
# Save all test probabilities/predictions for reproducibility.
prediction_frame = pd.DataFrame({'text': test_texts, 'gold_label': [TXT_CLASSES[i] for i in test_labels]})
for name, probs in all_test_probs.items():
    safe = re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
    prediction_frame[f'{safe}_prediction'] = [TXT_CLASSES[i] for i in probs.argmax(axis=1)]
    prediction_frame[f'{safe}_confidence'] = probs.max(axis=1)
    for j, cls in enumerate(TXT_CLASSES):
        prediction_frame[f'{safe}_p_{cls}'] = probs[:, j]
prediction_frame.to_csv(OUTPUT_DIR / 'test_predictions_all_models.csv', index=False)

# Recreate the original notebook's checkpoint envelope.
# `best_optimizer_state` corresponds to the selected best validation checkpoint.
compat_checkpoint = {
    'model': custom_model.state_dict(),
    'opt': best_optimizer_state,
    'step': int(best_step),
    'words': clf_words,
    'classes': TXT_CLASSES,
    'best_epoch': int(best_epoch),
    'validation_macro_f1': float(best_val_f1),
}
torch.save(compat_checkpoint, OUTPUT_DIR / 'clf_last.pt')
torch.save({
    'model': custom_model.state_dict(),
    'words': clf_words,
    'classes': TXT_CLASSES,
    'step': int(best_step),
    'best_epoch': int(best_epoch),
    'validation_macro_f1': float(best_val_f1),
    'test_metrics': metrics_for(test_labels, all_test_probs[custom_name]),
}, OUTPUT_DIR / 'clf_final.pt')

with open(OUTPUT_DIR / 'label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump({'classes': TXT_CLASSES, 'label_to_class': label_to_class, 'max_len': MAX_LEN,
               'max_vocab': MAX_VOCAB, 'best_epoch': int(best_epoch), 'best_step': int(best_step)}, f, indent=2)

print('Exported files to:', OUTPUT_DIR.resolve())
print('TTS-compatible checkpoint:', (OUTPUT_DIR / 'clf_last.pt').resolve())
print('Best epoch:', best_epoch, '| compatible step:', best_step, '| validation macro-F1:', round(best_val_f1, 4))
print('Copy clf_last.pt into your TTS CKPT folder, then set CLF_STEPS =', best_step, 'in the TTS notebook to prevent extra training.')

In [ ]:
%run -i classifier_research_extension.py